TOXIC COMMENT CLASSIFICATION

In [18]:

#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import string
import warnings 
warnings.filterwarnings('ignore')

#set nltk path
venv_data_path = os.path.join(os.getcwd(), 'venv', 'nltk_data')
if not os.path.exists(venv_data_path):
    os.makedirs(venv_data_path)

#Import NLP Libraries
import nltk
nltk.download('punkt',download_dir=venv_data_path)
nltk.download('punkt_tab', download_dir=venv_data_path)
nltk.download('stopwords',download_dir=venv_data_path)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.data.path.append(venv_data_path)


#Import Deep learing/ML libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,precision_score,recall_score
    ,f1_score,roc_auc_score
)
from sklearn.preprocessing import normalize
from collections import Counter

plt.style.use('seaborn-v0_8-whitegrid')
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


[nltk_data] Downloading package punkt to
[nltk_data]     e:\personal\MasterLiverpool\Assesment\CSCK507\Mid-
[nltk_data]     Module-Assignment\toxic\notebooks\venv\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     e:\personal\MasterLiverpool\Assesment\CSCK507\Mid-
[nltk_data]     Module-Assignment\toxic\notebooks\venv\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     e:\personal\MasterLiverpool\Assesment\CSCK507\Mid-
[nltk_data]     Module-Assignment\toxic\notebooks\venv\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Load Dataset

In [19]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print("===Before ETL===")
print(f"Training data shape: {train.shape}")
print(f"Test data shape: {test.shape}")


===Before ETL===
Training data shape: (159571, 8)
Test data shape: (153164, 2)


Initial EDA

In [20]:
#Baisc Info
print("===Basic Info===")
print(f"Training Sample: {len(train)}")
print(f"Features: {list(train.columns)}")

#Missing values In Raw Data
print("===Missing Values in Raw Data===")
print(train.isnull().sum())


===Basic Info===
Training Sample: 159571
Features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
===Missing Values in Raw Data===
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64


EDA: A: Number of sentences/tokens per class + check imbalance

In [ ]:
print("\n[EDA a] Sentence & Token Count Per Toxicity Class\n")

#Create Raw Count (required for avg tokens)
train["raw_word_count"] = train["comment_text"].astype(str).apply(lambda x: len(x.split()))

#Calculate each class average sentence length and token count
results  = []
for label in LABELS:
    class_data = train[train[label] == 1]
    sentence_count = len(class_data)
    total_tokens_count = class_data["raw_word_count"].sum()
    avg_tokens_count = round(class_data["raw_word_count"].mean(),1) if sentence_count > 0 else 0
        
    results.append({
        "Toxicity Class": label,
        "Sentence Count": sentence_count,
        "Total Tokens Count": total_tokens_count,
        "Average Tokens Count": avg_tokens_count
    })
    
    
#Display Table of Sentence & Token Count Per Toxicity Class
eda_table = pd.DataFrame(results)
print("EDA: Sentence & Token Count Per Toxicity Class")
print(eda_table.to_string(index=False))

# Check & print class imbalance
print("Dataset Imbalance Check:")
total_comments = len(train)
for res in results:
    cls = res["Toxicity Class"]
    count = res["sentence_count"]
    if count > 0:
        ratio = f"1 : {total_comments/ count: .0f}"
    else:
        ratio = "N/A"
        
    print(f" - {cls:15} -> {count:5} samples | Imbalance Ratio: {ratio}")
    
    
#Visualize Imbalance
plt.figure(figsize=(11,5))
plt.bar(eda_table["Toxicity Class"], eda_table["Sentence Count"], color='#ff6b6b')
plt.title("Sentence Count per Toxicity Class (Class Imbalance)")
plt.xticks(rotation=30)
plt.ylabel("Number of Sentence")
plt.tight_layout()
plt.show()


[EDA a] Sentence & Token Count Per Toxicity Class



AttributeError: 'dict' object has no attribute 'append'

ETL 

In [ ]:
#ETL clean text
stop_words = set(stopwords.words('english'))

def clean_text(text):
    try:
        text = str(text).strip()
        text = re.sub(r"http\S+|www\S+",'',text)
        text = re.sub(r"<.*?>",'',text )
        text = re.sub(r'[{}]'.format(re.escape(string.punctuation)), ' ',text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'\s+', ' ', text).strip() 
        
        tokens = word_tokenize(text)
        tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
        cleaned = ' '.join(tokens).strip()
        
        return cleaned if cleaned else np.nan
    except Exception as e:
        print(f"Handle Error: {e}")
        return np.nan
    
train['clean_text'] = train['comment_text'].apply(clean_text)
test['clean_text'] = test['comment_text'].apply(clean_text)
      
#Etl Handle Missing Value
#check missing value
print("===Handle Missing Data===")

missing_count = train["clean_text"].isnull().sum()
missing_pecentage = (missing_count / len(train)) * 100
print(f"Missing Values found NaN : {missing_count} cells, {missing_pecentage:.2f}%")

#If percenrtatge is less than 5%, we directly drop the missing value
if missing_pecentage > 5:
    raise ValueError("Critical Missing Value: More than 5% of the data is missing.")

train = train.dropna(subset=['clean_text'])
print(f"After dropping missing values, training data shape: {train.shape}")


===Handle Missing Data===
Missing Values found NaN : 66 cells, 0.04%
After dropping missing values, training data shape: (159505, 9)
